# Minimal Training + Reward Walkthrough for `verl-tool`

This notebook goes **one level beyond rollout validation**.

It ties together the smallest meaningful pieces:
- a **trajectory/rollout-like** object (`DataProto`)
- **reward computation** (token-level reward tensor)
- the **PPO update / checkpoint** code path (code evidence)

If you see import errors like `No module named 'torch'` or `No module named 'ray'`, run **Section 0** (deps install) and then restart the kernel.


## Section 0 — Minimal dependencies for this notebook

This notebook wants to execute a small **reward-tensor** demo.

- Required: `torch`, `tensordict`
- Note: `ray` is **not** installable on Python 3.14 today (no wheels). That blocks importing `verl.DataProto`.

So this notebook stays runnable by validating reward-tensor semantics without importing Ray.


In [1]:
from __future__ import annotations

import sys
import subprocess

needed = ["torch", "tensordict"]
missing = []
for m in needed:
    try:
        __import__(m)
    except Exception:
        missing.append(m)

print("python_version=", sys.version.split()[0])
print("Missing:", missing)

if missing:
    cmd = [sys.executable, "-m", "pip", "install", "-U", "torch", "tensordict"]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, text=True)
    if r.returncode != 0:
        raise SystemExit("Dependency install failed. See output above.")

    print("\nInstalled. Restart the kernel, then re-run from Section 1.")

# Ray note
try:
    import ray  # noqa: F401
    print("ray_import=OK")
except Exception as e:
    print("ray_import=FAILED (expected on Python 3.14)")
    print("ray_error=", repr(e))


python_version= 3.12.6
Missing: []


/Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-25 07:45:26,866	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


ray_import=OK


## Section 1 — Objective

Already validated:
- Tool server path works (`/get_observation`).
- A real agent-loop trajectory works (NL input → LLM action → tool → observation → next step).

What this notebook adds:
- Reward computation (token-level reward tensor semantics)
- Concrete code evidence for PPO update + checkpoints

Honesty / constraints:
- Full PPO is typically heavy (Ray + model engines). On Python 3.14, Ray is not installable today, so we do not run PPO here.
- We still make the update path clear by pointing to the exact code where backward/optimizer.step happen.

In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()

# Expect this notebook at: <repo>/verl-tool/scripts/test-rl-ppo/
VERL_TOOL_ROOT = HERE
for _ in range(6):
    if (VERL_TOOL_ROOT / "pyproject.toml").exists() and (VERL_TOOL_ROOT / "verl_tool").exists():
        break
    VERL_TOOL_ROOT = VERL_TOOL_ROOT.parent

REPO_ROOT = VERL_TOOL_ROOT.parent

print("python=", sys.executable)
print("VERL_TOOL_ROOT=", VERL_TOOL_ROOT)
print("REPO_ROOT=", REPO_ROOT)
assert (VERL_TOOL_ROOT / "verl_tool").exists(), "Could not locate verl-tool package root"
assert (VERL_TOOL_ROOT / "verl").exists(), "Could not locate embedded verl/ folder"

# Ensure local imports prefer the embedded `verl` code.
VERL_PKG_ROOT = VERL_TOOL_ROOT / "verl"
if str(VERL_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(VERL_PKG_ROOT))

print("sys.path[0]=", sys.path[0])


python= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv312/bin/python3
VERL_TOOL_ROOT= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool
REPO_ROOT= /Users/lzanda/Desktop/VRTOOL-Framework
sys.path[0]= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/verl


## One-page flow (trajectory → reward → PPO update)

This is the **end-to-end** conceptual flow in this repo, with the concrete code locations.

| Stage | What exists at this point | What happens | Where in code (repo) |
|---|---|---|---|
| 0. Input task | A user prompt / dataset item | Prepare prompt tokens + metadata | `verl_tool/trainer/main_ppo.py` (Hydra entry) + dataset configs under `verl_tool/trainer/config/` |
| 1. Rollout / trajectory | A rollout container (`DataProto`) with prompt + generated response + extra fields | The actor generates responses; in agentic mode it may do multi-turn tool calls | Agent loop: `verl_tool/agent_loop/agent_loop.py`, `verl_tool/agent_loop/verltool_agent_loop.py` (multi-turn + tool metrics) |
| 2. Env / tools | Tool server (`/get_observation`) | Execute tool actions and return observations appended into trajectory | Tool server: `verl_tool/servers/tool_server.py` + tools in `verl_tool/servers/tools/` |
| 3. Reward | Token-level reward tensor | Compute `reward_tensor` and attach it as `token_level_scores` | Reward entry: `verl_tool/trainer/ppo/reward.py` + reward managers: `verl_tool/workers/reward_manager/*` |
| 4. PPO prep | Batch now has `token_level_rewards` + advantages | Apply KL penalty (optional), compute advantages | Trainer loop: `verl/verl/trainer/ppo/ray_trainer.py` |
| 5. Update (weights change) | Actor/critic workers + optimizers | `update_actor` / `update_critic` → backward → optimizer step | `verl/verl/workers/roles/actor.py::update_actor` → `verl/verl/workers/engine/base.py::train_batch` → engine impl (e.g. `verl/verl/workers/engine/fsdp/transformer_impl.py`) |
| 6. Checkpoint | Model snapshots on disk | Save actor/critic checkpoints + dataloader state | `verl/verl/trainer/ppo/ray_trainer.py::_save_checkpoint` |

### What this notebook *verifies* vs *inspects*
- **Verifies (executes)**: reward-tensor semantics (token-level tensor with terminal reward placement).
- **Inspects (code evidence)**: where PPO consumes reward and where update/backward/checkpoint happen.


### Flow diagram (presentation-friendly)

```mermaid
flowchart LR
  A[Input task / prompt] --> B[Rollout / trajectory (DataProto)]
  B --> C[Actor generates response]
  C -->|tool-call action text| D[Tool server / environment]
  D -->|observation| B
  B --> E[Reward manager computes reward_tensor]
  E --> F[PPO trainer: token_level_scores/rewards + advantages]
  F --> G[Update: update_actor / update_critic]
  G --> H[Checkpoint save]
```


## Section 2 — Code path inspection

We’ll identify concrete files for:
- training entrypoint
- PPO trainer + update loop
- reward computation + reward managers
- rollout/trajectory objects (`DataProto`)
- checkpoint saving


In [3]:
paths = {
    # Entry + trainer integration
    "train_entry": VERL_TOOL_ROOT / "verl_tool/trainer/main_ppo.py",
    "verltool_ray_trainer": VERL_TOOL_ROOT / "verl_tool/trainer/ppo/ray_trainer.py",
    "verl_ray_trainer": VERL_TOOL_ROOT / "verl/verl/trainer/ppo/ray_trainer.py",

    # Reward
    "verltool_reward": VERL_TOOL_ROOT / "verl_tool/trainer/ppo/reward.py",
    "reward_managers_init": VERL_TOOL_ROOT / "verl_tool/workers/reward_manager/__init__.py",
    "reward_manager_simpletir": VERL_TOOL_ROOT / "verl_tool/workers/reward_manager/simple_tir.py",

    # Actor update + engine (where backward/step happens)
    "actor_worker": VERL_TOOL_ROOT / "verl/verl/workers/roles/actor.py",
    "engine_base": VERL_TOOL_ROOT / "verl/verl/workers/engine/base.py",
    "fsdp_engine_impl": VERL_TOOL_ROOT / "verl/verl/workers/engine/fsdp/transformer_impl.py",

    # Data structure for trajectories
    "protocol": VERL_TOOL_ROOT / "verl/verl/protocol.py",
}

for k, p in paths.items():
    print(f"{k:22s}", "OK" if p.exists() else "MISSING", "-", p.relative_to(REPO_ROOT))


train_entry            OK - verl-tool/verl_tool/trainer/main_ppo.py
verltool_ray_trainer   OK - verl-tool/verl_tool/trainer/ppo/ray_trainer.py
verl_ray_trainer       OK - verl-tool/verl/verl/trainer/ppo/ray_trainer.py
verltool_reward        OK - verl-tool/verl_tool/trainer/ppo/reward.py
reward_managers_init   OK - verl-tool/verl_tool/workers/reward_manager/__init__.py
reward_manager_simpletir OK - verl-tool/verl_tool/workers/reward_manager/simple_tir.py
actor_worker           OK - verl-tool/verl/verl/workers/roles/actor.py
engine_base            OK - verl-tool/verl/verl/workers/engine/base.py
fsdp_engine_impl       OK - verl-tool/verl/verl/workers/engine/fsdp/transformer_impl.py
protocol               OK - verl-tool/verl/verl/protocol.py


### Quick evidence snippets

We will print targeted snippets from the real code to show:
- reward is computed and assigned as `token_level_scores`
- update calls happen: `update_critic(...)` and `update_actor(...)`
- checkpoint save happens
- actor update calls `engine.train_batch(...)`, which does `zero_grad → backward → optimizer.step`


In [4]:
def show(path: Path, start: int, end: int) -> None:
    txt = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"--- {path.relative_to(REPO_ROOT)}:{start}-{end} ---")
    for i in range(start, min(end, len(txt)) + 1):
        print(f"{i:4d} {txt[i-1]}")
    print()

# (A) Trainer loop: reward → advantages → update actor/critic → save checkpoint
show(paths["verl_ray_trainer"], 1200, 1310)

# (B) Actor update path: update_actor calls engine.train_batch
show(paths["actor_worker"], 135, 205)

# (C) Engine.train_batch: zero_grad → forward_backward_batch → optimizer_step
show(paths["engine_base"], 90, 140)

# (D) Concrete backward + optimizer.step in FSDP engine impl (code evidence)
show(paths["fsdp_engine_impl"], 490, 550)


--- verl-tool/verl/verl/trainer/ppo/ray_trainer.py:1200-1310 ---
1200 
1201                         if "rollout_log_probs" in batch.batch.keys():
1202                             # TODO: we may want to add diff of probs too.
1203                             from verl.utils.debug.metrics import calculate_debug_metrics
1204 
1205                             metrics.update(calculate_debug_metrics(batch))
1206 
1207                     if self.use_reference_policy:
1208                         # compute reference log_prob
1209                         with marked_timer(str(Role.RefPolicy), timing_raw, color="olive"):
1210                             if not self.ref_in_actor:
1211                                 ref_log_prob = self.ref_policy_wg.compute_ref_log_prob(batch)
1212                             else:
1213                                 ref_log_prob = self.actor_rollout_wg.compute_ref_log_prob(batch)
1214                             batch = batch.union(ref_log_prob)
1215 
1216    

## Section 3 — Best minimal training target

**Chosen target (one best path):**

- **Reward-tensor validation + trainer/update inspection** (lightweight).
- We will:
  - build a minimal rollout-like batch (prompt + response + attention masks + ground_truth),
  - compute a **token-level reward tensor** (terminal reward on last response token),
  - then show exactly where that tensor is consumed in the PPO trainer (`token_level_scores → token_level_rewards → advantages → update_actor/update_critic`).

**Status:** partial execution.
- Reward-tensor computation is executed in this notebook.
- PPO update execution is shown via concrete code evidence (not run by default).

Note: On Python 3.14, `ray` is not installable today (no wheels), so importing `verl.DataProto` is blocked. The PPO flow is still verifiable from the code, and reward-tensor semantics are validated here.


## Section 4 — What is being trained?

Code evidence in this repo shows:
- The trainer calls **`critic_wg.update_critic(batch)`** (critic update) and **`actor_rollout_wg.update_actor(batch)`** (policy/actor update).
- `ActorWorker.update_actor(...)` calls `self.engine.train_batch(...)`.
- `BaseEngine.train_batch(...)` performs **`optimizer_zero_grad → forward_backward_batch → optimizer_step`**.
- In the FSDP engine impl, we can see `loss.backward()` and `self.optimizer.step()`.

What we do **not** fully validate here:
- Whether you are doing full fine-tuning vs LoRA/PEFT depends on the engine + config (`ActorConfig` / engine config). This notebook points to the file locations; running it end-to-end is heavier.
- Checkpoint format and exact model files depend on engine backend.


## Section 5 — Reward path

Concrete code path:
- Reward manager is loaded via `verl_tool/trainer/ppo/reward.py::load_reward_manager(...)`
- Reward is computed via `verl_tool/trainer/ppo/reward.py::compute_reward(data, reward_fn)`
- In the main PPO trainer loop (`verl/verl/trainer/ppo/ray_trainer.py`):
  - `reward_tensor` is assigned to `batch.batch["token_level_scores"]`
  - then converted to `token_level_rewards`
  - then advantages are computed
  - then actor/critic are updated.

Adding energy/latency/token costs:
- The cleanest place is to produce additional fields in the rollout (e.g. `verl_tool_metrics` / `tool_interact_info`)
  and then subtract penalties inside a reward manager.
- Conceptually:
  \(R = \alpha\,\text{accuracy} - \beta\,\text{latency} - \gamma\,\text{tokens} - \delta\,\text{energy}\)
- Practically: extend `reward_tensor[i, last_token]` and/or populate `reward_extra_info` for logging.


## Section 6 — Minimal execution plan

### Step 0 — Environment sanity
- **Goal**: confirm imports for embedded `verl` and `verl_tool`.
- **Validates**: local code is importable.

### Step 1 — Reward execution (lightweight)
- **Goal**: run a real reward manager on a minimal `DataProto`.
- **Validates**: reward manager produces a `reward_tensor` shaped like token-level scores.

### Step 2 — Update/checkpoint path evidence (code)
- **Goal**: point to exact code that does update + save.
- **Validates**: where to look for weight updates and checkpoints.

### Optional (heavier) — 1-step PPO smoke
- **Command** (not auto-run): `python3 -m verl_tool.trainer.main_ppo ...` with a tiny config.
- **Likely failures**: Ray setup, model engine requirements, GPU expectations.


## Section 7 — Notebook helper code

Helpers below:
- build a minimal `DataProto`
- run reward manager
- inspect the resulting reward tensor


In [5]:
import numpy as np
import torch

# NOTE:
# `verl.DataProto` imports `ray`, and Ray does not currently provide wheels for Python 3.14.
# To keep this notebook runnable in your current environment, we create a minimal
# DataProto-like carrier with the exact fields the reward code expects.


class MiniItem:
    def __init__(self, batch: dict, non_tensor_batch: dict):
        self.batch = batch
        self.non_tensor_batch = non_tensor_batch


class MiniDataProto:
    def __init__(self, batch: dict[str, torch.Tensor], non_tensor_batch: dict[str, np.ndarray]):
        self.batch = batch
        self.non_tensor_batch = non_tensor_batch
        self.meta_info = {}

    def __len__(self) -> int:
        key = next(iter(self.non_tensor_batch.keys()))
        return int(self.non_tensor_batch[key].shape[0])

    def __getitem__(self, i: int) -> MiniItem:
        batch_i = {k: v[i] for k, v in self.batch.items()}
        non_tensor_i = {k: v[i] for k, v in self.non_tensor_batch.items()}
        return MiniItem(batch=batch_i, non_tensor_batch=non_tensor_i)


class CharTokenizer:
    """Tiny tokenizer to avoid downloading HF models."""

    def encode(self, s: str) -> list[int]:
        s = s or ""
        return [max(1, min(255, ord(ch))) for ch in s]

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        return "".join(chr(int(x)) for x in ids if int(x) != 0)


def pad_to(t: torch.Tensor, length: int) -> torch.Tensor:
    if t.numel() >= length:
        return t[:length]
    return torch.cat([t, torch.zeros(length - t.numel(), dtype=t.dtype)], dim=0)


def make_minimal_batch(prompt: str, response: str, *, ground_truth: str, data_source: str = "demo") -> MiniDataProto:
    tok = CharTokenizer()
    prompt_ids = torch.tensor(tok.encode(prompt), dtype=torch.long)
    response_ids = torch.tensor(tok.encode(response), dtype=torch.long)

    max_prompt = int(prompt_ids.numel())
    max_resp = int(response_ids.numel())

    prompts = pad_to(prompt_ids, max_prompt).unsqueeze(0)       # (1, P)
    responses = pad_to(response_ids, max_resp).unsqueeze(0)     # (1, R)

    attention_mask = torch.cat(
        [torch.ones((1, max_prompt), dtype=torch.long), torch.ones((1, max_resp), dtype=torch.long)],
        dim=1,
    )

    non_tensors = {
        "data_source": np.array([data_source], dtype=object),
        "reward_model": np.array([{"ground_truth": ground_truth}], dtype=object),
    }

    return MiniDataProto(
        batch={"prompts": prompts, "responses": responses, "attention_mask": attention_mask},
        non_tensor_batch=non_tensors,
    )


dp = make_minimal_batch(
    prompt="Compute 12345*6789.",
    response="The answer is \\boxed{83810205}.",
    ground_truth="83810205",
)
print("MiniDataProto len:", len(dp))
print("batch keys:", list(dp.batch.keys()))
print("non_tensor_batch keys:", list(dp.non_tensor_batch.keys()))


MiniDataProto len: 1
batch keys: ['prompts', 'responses', 'attention_mask']
non_tensor_batch keys: ['data_source', 'reward_model']


## Section 8 — Visible trajectory + reward example

On your current Python 3.14 environment, we cannot import the full `verl_tool` reward managers because they depend on `verl.DataProto` → `ray` (Ray wheels are not available).

So this section validates the **same reward tensor semantics** the repo uses:
- reward is a **token-level tensor**
- the terminal reward is placed at the **last valid response token**

We also keep the PPO training/update path verifiable via concrete code snippets above.


In [6]:
# Reward execution (minimal, runnable on Python 3.14)
#
# We cannot import the full `verl_tool` reward managers here because they depend on
# `verl.DataProto`, and that imports `ray` (Ray wheels not available for Python 3.14).
#
# Instead, we validate the *same reward-tensor semantics used by the repo*:
# - token-level reward tensor with terminal reward at last valid response token
#
# Code evidence for the terminal reward assignment lives in:
# - `verl_tool/workers/reward_manager/simple_tir.py` (sets `reward_tensor[i, valid_response_length - 1] = reward`)

import re


def extract_boxed_answer(text: str) -> str:
    m = re.findall(r"\\\\boxed\{([^{}]*)\}", text or "")
    return m[-1] if m else ""


def compute_score(solution_str: str, ground_truth: str) -> float:
    # Minimal scoring: +1 if extracted boxed answer matches, else -1
    pred = extract_boxed_answer(solution_str)
    return 1.0 if pred == str(ground_truth) else -1.0


# Build reward tensor like the reward managers do: zeros except terminal token.
responses = dp.batch["responses"]
reward_tensor = torch.zeros_like(responses, dtype=torch.float32)

# For this demo we know all tokens are valid (attention_mask=1).
# In the real code, valid_response_length is computed from attention_mask slicing.
valid_response_length = int(responses.shape[1])
terminal_idx = max(0, valid_response_length - 1)

# Decode response string for scoring
tok = CharTokenizer()
response_ids = responses[0]
response_str = tok.decode(response_ids)

reward = compute_score(response_str, dp.non_tensor_batch["reward_model"][0]["ground_truth"])
reward_tensor[0, terminal_idx] = reward

print("response_str:", response_str)
print("ground_truth:", dp.non_tensor_batch["reward_model"][0]["ground_truth"])
print("terminal_idx:", terminal_idx)
print("reward:", reward)
print("reward_tensor shape:", tuple(reward_tensor.shape))
print("nonzero:", (reward_tensor != 0).nonzero(as_tuple=False).tolist())


response_str: The answer is \boxed{83810205}.
ground_truth: 83810205
terminal_idx: 30
reward: -1.0
reward_tensor shape: (1, 31)
nonzero: [[0, 30]]


## Section 9 — Training/update evidence

### Do weights get updated?
Code evidence (not executed here):
- `verl/verl/trainer/ppo/ray_trainer.py` calls:
  - `critic_wg.update_critic(batch)`
  - `actor_rollout_wg.update_actor(batch)`
- `verl/verl/workers/roles/actor.py::ActorWorker.update_actor` calls:
  - `self.engine.train_batch(mini_batch, self.loss_fn)`
- `verl/verl/workers/engine/base.py::BaseEngine.train_batch` does:
  - `optimizer_zero_grad()`
  - `forward_backward_batch(..., forward_only=False)`
  - `optimizer_step()`
- `verl/verl/workers/engine/fsdp/transformer_impl.py` shows concrete:
  - `loss.backward()`
  - `self.optimizer.step()`

### Checkpoint evidence
- `verl/verl/trainer/ppo/ray_trainer.py::_save_checkpoint` calls:
  - `actor_rollout_wg.save_checkpoint(...)`
  - `critic_wg.save_checkpoint(...)` (if critic enabled)

What to look for when you run real training later:
- a `default_local_dir/global_step_*/` folder
- actor/critic subfolders
- `latest_checkpointed_iteration.txt`


## Section 10 — Final short readout

**Validated in this notebook (executed):**
- `DataProto` object creation (trajectory-like carrier for prompt/response + metadata)
- Reward manager execution producing a **token-level `reward_tensor`**

**Validated in this notebook (code evidence, not executed):**
- Where reward enters PPO: `token_level_scores → token_level_rewards → advantages`
- Where update happens: `update_actor` / `update_critic`
- Where backward + optimizer step happens: `BaseEngine.train_batch` + FSDP impl (`loss.backward`, `optimizer.step`)
- Where checkpoints are saved: `_save_checkpoint`

**Unverified (by design):**
- Running a full PPO step with real model engines (Ray + FSDP/Megatron/vLLM) on this laptop.

**Next experiment:**
- Build a tiny training config that runs **1–2 steps** with a very small local model backend,
  then confirm:
  - checkpoint folder is created,
  - actor weights change (hash / parameter diff),
  - reward contains your energy/latency/token penalties.
